# 01 — Limpieza y Preparación de Datos

**Objetivo:** Explorar, diagnosticar y limpiar los 6 datasets del proyecto climático y dejarlos listos para el análisis exploratorio.

## Decisiones de limpieza documentadas
- `indicators`: pivot de formato ancho a largo, filtro año ≥ 2000, corrección ISO2 Namibia
- `temp_monthly`: creación de columna datetime, separación global vs regiones
- `co2`, `energy`, `carbon_prices`, `events`: sin transformaciones necesarias

## Output
7 archivos `.csv` limpios guardados en `/data/clean/`

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

ruta_datos = '/content/drive/MyDrive/Proyecto_Clima/ data/'
print(os.listdir(ruta_datos))

['temperature_anomaly_monthly.csv', 'climate_events.csv', 'co2_emissions_yearly.csv', 'energy_mix_yearly.csv', 'carbon_prices_daily.csv', 'climate_change_indicators.csv']


In [3]:
import pandas as pd

df_indicators = pd.read_csv(ruta_datos + 'climate_change_indicators.csv',
                             keep_default_na=False,
                             na_values=['', ' ', 'NULL', 'null', 'N/A'])
df_temp_monthly = pd.read_csv(ruta_datos + 'temperature_anomaly_monthly.csv')
df_co2 = pd.read_csv(ruta_datos + 'co2_emissions_yearly.csv')
df_energy = pd.read_csv(ruta_datos + 'energy_mix_yearly.csv')
df_carbon_prices = pd.read_csv(ruta_datos + 'carbon_prices_daily.csv')
df_events = pd.read_csv(ruta_datos + 'climate_events.csv')

print("✅ Todos los datasets cargados correctamente")

✅ Todos los datasets cargados correctamente


In [4]:
# Dimenciones y tipos de datos
dataframes = {
    'indicators': df_indicators,
    'temp_monthly': df_temp_monthly,
    'co2': df_co2,
    'energy': df_energy,
    'carbon_prices': df_carbon_prices,
    'events': df_events
}

for nombre, df in dataframes.items():
    print(f"\n{'='*50}")
    print(f"📊 {nombre.upper()}")
    print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
    print(df.dtypes.value_counts())


📊 INDICATORS
Filas: 225 | Columnas: 72
float64    62
object      9
int64       1
Name: count, dtype: int64

📊 TEMP_MONTHLY
Filas: 2528 | Columnas: 7
object     3
int64      2
float64    2
Name: count, dtype: int64

📊 CO2
Filas: 1350 | Columnas: 8
float64    4
object     3
int64      1
Name: count, dtype: int64

📊 ENERGY
Filas: 1350 | Columnas: 14
float64    10
object      3
int64       1
Name: count, dtype: int64

📊 CARBON_PRICES
Filas: 15866 | Columnas: 5
object     3
int64      1
float64    1
Name: count, dtype: int64

📊 EVENTS
Filas: 50 | Columnas: 11
int64     6
object    5
Name: count, dtype: int64


In [5]:
# Valores nulos
for nombre, df in dataframes.items():
    nulos = df.isnull().sum()
    nulos = nulos[nulos > 0]
    if len(nulos) > 0:
        print(f"\n⚠️ {nombre.upper()} tiene valores nulos:")
        print(nulos)
    else:
        print(f"\n✅ {nombre.upper()} no tiene nulos")


⚠️ INDICATORS tiene valores nulos:
ISO2      1
F1961    37
F1962    36
F1963    37
F1964    37
         ..
F2018    12
F2019    12
F2020    13
F2021    12
F2022    12
Length: 63, dtype: int64

⚠️ TEMP_MONTHLY tiene valores nulos:
co2_ppm    2212
dtype: int64

✅ CO2 no tiene nulos

✅ ENERGY no tiene nulos

✅ CARBON_PRICES no tiene nulos

✅ EVENTS no tiene nulos


In [6]:
# Duplicados
for nombre, df in dataframes.items():
    duplicados = df.duplicated().sum()
    print(f"{nombre}: {duplicados} filas duplicadas")

indicators: 0 filas duplicadas
temp_monthly: 0 filas duplicadas
co2: 0 filas duplicadas
energy: 0 filas duplicadas
carbon_prices: 0 filas duplicadas
events: 0 filas duplicadas


In [7]:
print(df_indicators[df_indicators['ISO2'].isnull()][['Country', 'ISO2']])

    Country ISO2
221   World  NaN


In [8]:
print(df_temp_monthly[df_temp_monthly['co2_ppm'].notna()]['region'].value_counts())

region
Global    316
Name: count, dtype: int64


In [9]:
# Vista previa
df_indicators.head()

,ObjectId,Country,ISO2,ISO3,Indicator,Unit,Source,CTS_Code,CTS_Name,CTS_Full_Descriptor,...,F2013,F2014,F2015,F2016,F2017,F2018,F2019,F2020,F2021,F2022
0,1,"Afghanistan, Islamic Rep. of",AF,AFG,Temperature change with respect to a baseline ...,Degree Celsius,Food and Agriculture Organization of the Unite...,ECCS,Surface Temperature Change,"Environment, Climate Change, Climate Indicator...",...,1.281,0.456,1.093,1.555,1.540,1.544,0.910,0.498,1.327,2.012
1,2,Albania,AL,ALB,Temperature change with respect to a baseline ...,Degree Celsius,Food and Agriculture Organization of the Unite...,ECCS,Surface Temperature Change,"Environment, Climate Change, Climate Indicator...",...,1.333,1.198,1.569,1.464,1.121,2.028,1.675,1.498,1.536,1.518
2,3,Algeria,DZ,DZA,Temperature change with respect to a baseline ...,Degree Celsius,Food and Agriculture Organization of the Unite...,ECCS,Surface Temperature Change,"Environment, Climate Change, Climate Indicator...",...,1.192,1.690,1.121,1.757,1.512,1.210,1.115,1.926,2.330,1.688
3,4,American Samoa,AS,ASM,Temperature change with respect to a baseline ...,Degree Celsius,Food and Agriculture Organization of the Unite...,ECCS,Surface Temperature Change,"Environment, Climate Change, Climate Indicator...",...,1.257,1.170,1.009,1.539,1.435,1.189,1.539,1.430,1.268,1.256
4,5,"Andorra, Principality of",AD,AND,Temperature change with respect to a baseline ...,Degree Celsius,Food and Agriculture Organization of the Unite...,ECCS,Surface Temperature Change,"Environment, Climate Change, Climate Indicator...",...,0.831,1.946,1.690,1.990,1.925,1.919,1.964,2.562,1.533,3.243


In [10]:
df_temp_monthly.head()

,year_month,year,month,region,temp_anomaly_c,temp_anomaly_baseline,co2_ppm
0,2000-01,2000,1,Global,0.432,1951-1980,367.44
1,2000-01,2000,1,Arctic,1.500,1951-1980,NaN
2,2000-01,2000,1,Northern_Hemisphere,0.468,1951-1980,NaN
3,2000-01,2000,1,Southern_Hemisphere,0.320,1951-1980,NaN
4,2000-01,2000,1,Tropics,0.272,1951-1980,NaN


In [11]:
df_co2.head()

,year,country,iso3,region,co2_emissions_mt,population_millions,co2_per_capita_t,co2_intensity_kg_per_gdp_usd
0,2000,China,CHN,Asia,3535.5,1198.5,2.95,0.59
1,2001,China,CHN,Asia,4049.2,1207.9,3.35,0.61
2,2002,China,CHN,Asia,4601.0,1217.4,3.78,0.63
3,2003,China,CHN,Asia,5121.6,1226.8,4.17,0.64
4,2004,China,CHN,Asia,5553.3,1236.3,4.49,0.64


In [12]:
df_energy.head()

,year,country,iso3,region,coal_pct,oil_pct,gas_pct,nuclear_pct,hydro_pct,solar_pct,wind_pct,other_renewables_pct,renewables_total_pct,fossil_total_pct
0,2000,China,CHN,Asia,68.00,21.98,2.78,0.87,6.36,0.01,0.00,0.00,6.37,92.75
1,2001,China,CHN,Asia,67.64,21.35,3.36,0.97,6.46,0.00,0.04,0.17,6.68,92.36
2,2002,China,CHN,Asia,67.56,21.21,3.19,1.30,6.38,0.00,0.35,0.03,6.76,91.95
3,2003,China,CHN,Asia,67.69,20.71,3.24,1.30,6.33,0.07,0.33,0.34,7.06,91.64
4,2004,China,CHN,Asia,67.76,20.44,3.48,1.02,6.57,0.02,0.20,0.50,7.29,91.68


In [13]:
df_carbon_prices.head()

,date,year,market,currency,price
0,2005-04-22,2005,EU_ETS,EUR,7.17
1,2005-04-25,2005,EU_ETS,EUR,7.13
2,2005-04-26,2005,EU_ETS,EUR,7.44
3,2005-04-27,2005,EU_ETS,EUR,7.38
4,2005-04-28,2005,EU_ETS,EUR,7.38


In [14]:
df_events.head()

,event_id,date,year,month,region,event_type,severity_score,description,is_policy,is_extreme_weather,is_disaster
0,CE001,2003-08-01,2003,8,Europe,heatwave,8,"European heatwave kills 70,000+",0,1,0
1,CE002,2004-12-26,2004,12,Asia,tsunami,9,"Indian Ocean tsunami (227,000 deaths)",0,0,1
2,CE003,2005-08-29,2005,8,USA,hurricane,9,"Hurricane Katrina (~1,800 deaths, $125B)",0,1,0
3,CE004,2008-05-12,2008,5,China,earthquake,8,Sichuan earthquake,0,0,1
4,CE005,2009-12-07,2009,12,global,policy,8,Copenhagen Climate Conference COP15,1,0,0


In [15]:
# Pivot de indicators (de ancho a largo)
# Seleccionamos las columnas de identificación y solo años desde 2000
cols_id = ['Country', 'ISO2', 'ISO3', 'Indicator', 'Unit']
cols_años = [col for col in df_indicators.columns if col.startswith('F2')]

df_indicators_long = df_indicators[cols_id + cols_años].melt(
    id_vars=cols_id,
    value_vars=cols_años,
    var_name='Year',
    value_name='Value'
)

# Limpiar el año: quitar la 'F' y convertir a entero
df_indicators_long['Year'] = df_indicators_long['Year'].str.replace('F', '').astype(int)

# Eliminar filas sin valor
df_indicators_long = df_indicators_long.dropna(subset=['Value'])

print(f"Filas después del pivot: {df_indicators_long.shape[0]:,}")
print(df_indicators_long.head())

Filas después del pivot: 4,910
                        Country ISO2 ISO3  \
0  Afghanistan, Islamic Rep. of   AF  AFG   
1                       Albania   AL  ALB   
2                       Algeria   DZ  DZA   
3                American Samoa   AS  ASM   
4      Andorra, Principality of   AD  AND   

                                           Indicator            Unit  Year  \
0  Temperature change with respect to a baseline ...  Degree Celsius  2000   
1  Temperature change with respect to a baseline ...  Degree Celsius  2000   
2  Temperature change with respect to a baseline ...  Degree Celsius  2000   
3  Temperature change with respect to a baseline ...  Degree Celsius  2000   
4  Temperature change with respect to a baseline ...  Degree Celsius  2000   

   Value  
0  0.993  
1  1.065  
2  0.820  
3  0.626  
4  1.050  


In [19]:
#Revisamos que indicadores hay
print(df_indicators_long['Indicator'].value_counts())

Indicator
Temperature change with respect to a baseline climatology, corresponding to the period 1951-1980    4910
Name: count, dtype: int64


In [20]:
df_indicators_long['Indicator'] = 'Temperature Change (°C)'
print(df_indicators_long['Indicator'].value_counts())

Indicator
Temperature Change (°C)    4910
Name: count, dtype: int64


In [21]:
#Limpiamos temp_monthly
# Convertir Month a datetime
df_temp_monthly['date'] = pd.to_datetime(
    df_temp_monthly['year'].astype(str) + '-' + df_temp_monthly['month'].astype(str),
    format='%Y-%m'
)

# Separar dataset global (con co2_ppm) del resto
df_temp_global = df_temp_monthly[df_temp_monthly['region'] == 'Global'].copy()
df_temp_regions = df_temp_monthly[df_temp_monthly['region'] != 'Global'].copy()

print(f"Global: {df_temp_global.shape[0]} filas")
print(f"Regiones: {df_temp_regions.shape[0]} filas")
print(df_temp_global.head(3))

Global: 316 filas
Regiones: 2212 filas
   year_month  year  month  region  temp_anomaly_c temp_anomaly_baseline  \
0     2000-01  2000      1  Global           0.432             1951-1980   
8     2000-02  2000      2  Global           0.451             1951-1980   
16    2000-03  2000      3  Global           0.428             1951-1980   

    co2_ppm       date  
0    367.44 2000-01-01  
8    369.21 2000-02-01  
16   371.01 2000-03-01  


In [22]:
# Verificar columnas
print(df_co2.columns.tolist())
print(df_co2.head(3))

['year', 'country', 'iso3', 'region', 'co2_emissions_mt', 'population_millions', 'co2_per_capita_t', 'co2_intensity_kg_per_gdp_usd']
   year country iso3 region  co2_emissions_mt  population_millions  \
0  2000   China  CHN   Asia            3535.5               1198.5   
1  2001   China  CHN   Asia            4049.2               1207.9   
2  2002   China  CHN   Asia            4601.0               1217.4   

   co2_per_capita_t  co2_intensity_kg_per_gdp_usd  
0              2.95                          0.59  
1              3.35                          0.61  
2              3.78                          0.63  


In [23]:
# Limpiar columnas innecesarias de temp_monthly
df_temp_global = df_temp_global.drop(columns=['year_month', 'temp_anomaly_baseline'])
df_temp_regions = df_temp_regions.drop(columns=['year_month', 'temp_anomaly_baseline', 'co2_ppm'])

print(df_temp_global.columns.tolist())
print(df_temp_regions.columns.tolist())

['year', 'month', 'region', 'temp_anomaly_c', 'co2_ppm', 'date']
['year', 'month', 'region', 'temp_anomaly_c', 'date']


In [24]:
# Guardar datasets limpios
ruta_clean = ruta_datos + 'clean/'
import os
os.makedirs(ruta_clean, exist_ok=True)

df_indicators_long.to_csv(ruta_clean + 'indicators_clean.csv', index=False)
df_temp_global.to_csv(ruta_clean + 'temp_global_clean.csv', index=False)
df_temp_regions.to_csv(ruta_clean + 'temp_regions_clean.csv', index=False)
df_co2.to_csv(ruta_clean + 'co2_clean.csv', index=False)
df_energy.to_csv(ruta_clean + 'energy_clean.csv', index=False)
df_carbon_prices.to_csv(ruta_clean + 'carbon_prices_clean.csv', index=False)
df_events.to_csv(ruta_clean + 'events_clean.csv', index=False)

print("✅ Todos los datasets limpios guardados en /data/clean/")

✅ Todos los datasets limpios guardados en /data/clean/


In [25]:
# Resumen Final de la limpieza
datasets_clean = {
    'indicators_long': df_indicators_long,
    'temp_global': df_temp_global,
    'temp_regions': df_temp_regions,
    'co2': df_co2,
    'energy': df_energy,
    'carbon_prices': df_carbon_prices,
    'events': df_events
}

print("RESUMEN FINAL — DATASETS LIMPIOS")
print("="*45)
for nombre, df in datasets_clean.items():
    print(f"✅ {nombre}: {df.shape[0]:,} filas × {df.shape[1]} columnas")

RESUMEN FINAL — DATASETS LIMPIOS
✅ indicators_long: 4,910 filas × 7 columnas
✅ temp_global: 316 filas × 6 columnas
✅ temp_regions: 2,212 filas × 5 columnas
✅ co2: 1,350 filas × 8 columnas
✅ energy: 1,350 filas × 14 columnas
✅ carbon_prices: 15,866 filas × 5 columnas
✅ events: 50 filas × 11 columnas
